<a href="https://colab.research.google.com/github/Noisy77-pixel/urdu-ocr-codesaviours-si26-bilal/blob/main/SI26_Week2_Bilal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Gap Analysis — Tesseract vs Actual Text

### Comparison Table

| Image | Actual Text | Tesseract Output | What Went Wrong |
|-------|------------|------------------|-----------------|
| news_screenshot_001.png | ایران کی امریکی حملوں میں 50 افراد کی ہلاکت کی تصدیق | میں کسی کی ڈکٹیشن نہیں لوں گا... بحران جب صدر اور وزیر ا | Picked up wrong headline, cut off mid-word |
| news_screenshot_002.png | فیچر اور تجزیے | فیچر اور تجزیے ✓ (but rest garbled: 1" --× ) | Got heading right but gibberish symbols mixed in |
| news_screenshot_003.png | پاکستان میں ڈیزل 31.05 اور پٹرول 5.44 روپے مہنگا | بالواسطہ ہراسانی پر پانچ لاکھ روپے... | Completely different text detected — wrong region of page |
| news_screenshot_004.png | پاکستان کے خلاف 365 رنز کی ناقابل شکست اننگز | 75 7۸ہ اخدتدننت... 53077123157773 | Mostly garbled numbers and broken characters |
| news_screenshot_005.png | زیارت حملے کے خلاف دھرنا ختم | کشمیر کی وادی ہماری ہے... | Read a different article entirely |

### Summary

**Tesseract fails on Urdu because** it was primarily designed for Latin-based scripts and struggles with the fundamental characteristics of Urdu text. The Nastaliq script's cursive, connected nature means characters flow into each other and change shape based on position — Tesseract's segmentation algorithms cannot reliably identify character boundaries. The right-to-left direction causes word and line ordering errors. Diacritical marks (dots above/below characters) are frequently misread or ignored, turning one character into a completely different one. On full-page screenshots with mixed content (images + text + headlines), Tesseract fails to correctly isolate text regions, often reading the wrong section entirely or producing garbled numbers instead of text. This is why we need to fine-tune a deep learning model (TrOCR) specifically trained on our Urdu dataset.

In [2]:
!pip install opencv-python-headless pillow matplotlib
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [3]:
def preprocess_image(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return None

    # Step 1: Convert to grayscale (removes colour noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize to standard size (keeps all images same dimensions)
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Remove noise (makes text cleaner)
    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    # Step 4: Binarise (make pixels either pure black or pure white)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    # Save processed image
    cv2.imwrite(save_path, binary)
    return binary
# Create output folder
os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [9]:
def visualise_preprocessing(image_path):
    """Show each preprocessing step side-by-side for understanding."""
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (512, 128))
    denoised = cv2.fastNlMeansDenoising(resized, h=10)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    titles = ['Original (Gray)', 'Resized', 'Denoised', 'Binarised']
    images = [gray, resized, denoised, binary]

    for ax, title, image in zip(axes, titles, images):
        ax.imshow(image, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
# Uncomment to visualise on a sample image:
# visualise_preprocessing('data/synthetic/urdu_1.png')
print('Visualisation function ready! Uncomment the line above to test.')

Visualisation function ready! Uncomment the line above to test.


In [10]:
import glob
# Find all images in data/raw/ and data/synthetic/
all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)
all_images += glob.glob('data/raw/**/*.jpeg', recursive=True)
all_images += glob.glob('data/synthetic/*.png', recursive=True)
print(f'Found {len(all_images)} images to process')
processed_count = 0
failed_count = 0
for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1
    else:
        failed_count += 1
print(f'\nDone! Processed {processed_count} images')
if failed_count > 0:
    print(f'Failed to process {failed_count} images')
print('Check data/processed/ folder')

Found 404 images to process

Done! Processed 404 images
Check data/processed/ folder


In [13]:
# !apt-get install -y tesseract-ocr tesseract-ocr-urd
# !pip install pytesseract
import pytesseract
from PIL import Image
import glob
# Test on 5 of your processed images
test_images = list(glob.glob('data/processed/*.png'))[:5]
if not test_images:
    print('No processed images found! Run the preprocessing cells first.')
else:
    print('=== Tesseract Results on Urdu Images ===')
    print()

    results = []
    for img_path in test_images:
        img = Image.open(img_path)
        # 'urd' tells Tesseract to use the Urdu language model
        result = pytesseract.image_to_string(img, lang='urd')

        print(f'Image: {os.path.basename(img_path)}')
        print(f'Tesseract output: {result.strip() if result.strip() else "(empty/no text detected)"}')
        print('---')

        results.append({
            'image': os.path.basename(img_path),
            'tesseract_output': result.strip()
        })


=== Tesseract Results on Urdu Images ===

Image: hw_char_185.png
Tesseract output: (empty/no text detected)
---
Image: hw_char_168.png
Tesseract output: (empty/no text detected)
---
Image: hw_char_107.png
Tesseract output: (empty/no text detected)
---
Image: ravi_040.png
Tesseract output: (empty/no text detected)
---
Image: ravi_001.png
Tesseract output: (empty/no text detected)
---


In [14]:
import pytesseract
from PIL import Image
import glob, os

# Test specifically on news screenshots (actual Urdu text)
test_images = sorted(glob.glob('data/processed/news_screenshot_*.png'))[:5]

if not test_images:
    # Fallback: try raw newspaper images if processed ones aren't named right
    test_images = sorted(glob.glob('data/raw/newspaper/news_screenshot_*.png'))[:5]

print('=== Tesseract Results on Urdu News Screenshots ===\n')

for img_path in test_images:
    img = Image.open(img_path)
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {os.path.basename(img_path)}')
    print(f'Tesseract output: {result.strip() if result.strip() else "(empty/no text detected)"}')
    print('---')

=== Tesseract Results on Urdu News Screenshots ===

Image: news_screenshot_001.png
Tesseract output: (empty/no text detected)
---
Image: news_screenshot_002.png
Tesseract output: (empty/no text detected)
---
Image: news_screenshot_003.png
Tesseract output: (empty/no text detected)
---
Image: news_screenshot_004.png
Tesseract output: (empty/no text detected)
---
Image: news_screenshot_005.png
Tesseract output: (empty/no text detected)
---


In [15]:
import pytesseract
from PIL import Image
import glob, os

# Test on RAW screenshots (before preprocessing squishes them)
test_images = sorted(glob.glob('data/raw/newspaper/news_screenshot_*.png'))[:5]

print('=== Tesseract Results on RAW Urdu News Screenshots ===\n')

for img_path in test_images:
    img = Image.open(img_path)
    print(f'Image: {os.path.basename(img_path)} (size: {img.size})')
    result = pytesseract.image_to_string(img, lang='urd')
    output = result.strip()
    if output:
        # Show first 200 chars to keep it readable
        print(f'Tesseract output:\n{output[:200]}')
    else:
        print('Tesseract output: (empty/no text detected)')
    print('---')

=== Tesseract Results on RAW Urdu News Screenshots ===

Image: news_screenshot_001.png (size: (1206, 891))
Tesseract output:
پاکستان | آس پاس |ورلڈ ' کھیل | فن فنکار سائنس

       

"میں کسی کی ڈکٹیشن نہیں لوں گا 'کوہ کلنگ*: ٹرمپ کی نظر ایران کے
سے استعفے تک: 1993 کا سیاسی اس پراسرار پہاڑ پر کیوں ہے؟
بحران جب صدر اور وزیر ا
---
Image: news_screenshot_002.png (size: (1021, 661))
Tesseract output:
فیچر اور تجزیے
1" --×

  

'
پاکستان کے خلاف 365 رنز کی ناقابل
شکست اننگز کھیلنے والے سر گیری
سوبرز جو کسی بھی کپتان کا خواب
تھے

8 جولائی 2026

07ر بد 7ت7

میں جو آپ کے لیے معنی رکھتی ہیں

سبسکرائب ک
---
Image: news_screenshot_003.png (size: (1033, 643))
Tesseract output:
بالواسطہ ہراسانی پر پانچ لاکھ روپے ب ِ ن دہانی زیارت حملہ: آخری فون کال, لاشوں ‏ "آج شام چار بجے تمھیں قتل کر دیں

جرمانہ 'واٹئس ایپ گروپ میں بانج ہ کی واپسی میں مشکلات اور کمک ‏ گے ایک انگریز بھی کی 
---
Image: news_screenshot_004.png (size: (1311, 911))
Tesseract output:
75 7۸ہ اخدتدننت

2026 ,70017:18 528
ا 7ب صدنعدد دہ
2ا